# Chương 4 – In-Context Learning, Few-Shot và RAG (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phuongtnd1994/ptnd-ModelAdaptationBook/blob/main/code/chapter04/chapter04_colab.ipynb)

Các listing 4.1–4.5: few-shot, many-shot, prompt validator, RAG tối giản và đánh giá RAG.
Phần lớn chạy trên CPU (backend `mock`/`hash`/`st`); backend `hf` dùng GPU để chạy Qwen3-4B.
Phần bonus thay LLM giả (`stub_llm`) của RAG bằng **Claude**.

## 0. Cài đặt môi trường

Chạy lần lượt các ô dưới đây **một lần** mỗi khi mở runtime mới. Ô đầu tiên clone repo, cài package của sách và tạo hàm `run()` để chạy lệnh và in log trực tiếp ra notebook.

> Nếu pip báo `ERROR: pip's dependency resolver ...` về `gradio`, `diffusers`, `huggingface-hub` hoặc `safetensors`: **bỏ qua được**. Đó là các thư viện Colab cài sẵn mà sách không dùng; sách cố ý giữ `transformers<5` và `safetensors<0.6`.

In [ ]:
import os, re, sys, subprocess

# Repo chứa code của sách. Nếu fork của bạn để private, đổi sang repo gốc:
#   https://github.com/bahree/ModelAdaptationBook.git
REPO_URL = "https://github.com/phuongtnd1994/ptnd-ModelAdaptationBook.git"
REPO_DIR = "/content/ModelAdaptationBook"
CODE_DIR = f"{REPO_DIR}/code"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

os.environ["PYTHONUNBUFFERED"] = "1"   # log của script con hiện ra ngay
os.environ["WANDB_DISABLED"] = "true"  # không dùng Weights & Biases trên Colab
# Sách chỉ dùng PyTorch. Không cho transformers nạp TensorFlow có sẵn trên Colab
# (tránh cảnh báo giả "Could not find cuda drivers" của TensorFlow).
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


def run(cmd: str) -> None:
    """Chạy một lệnh shell từ thư mục code/, in log trực tiếp, báo lỗi nếu thất bại."""
    # Một số script không tự tạo thư mục output -> tạo sẵn trước khi chạy.
    for flag, path in re.findall(r"--(output_dir|output|out)\s+[\"']?([^\s\"']+)", cmd):
        target = path if flag == "output_dir" or not os.path.splitext(path)[1] else os.path.dirname(path)
        if target:
            os.makedirs(os.path.join(CODE_DIR, target), exist_ok=True)
    print(f"$ {cmd}\n", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE_DIR, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {p.returncode}): {cmd}")


run('pip install -q -e ".[dev]"')
print("\nĐã cài xong. Thư mục làm việc:", os.getcwd())

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} | {mem:.1f} GB | compute capability {cap[0]}.{cap[1]}")
    if cap < (8, 0):
        print("CẢNH BÁO: GPU này (ví dụ T4) không hỗ trợ bf16 gốc. Các bước TRAINING "
              "trong sách dùng bf16 -> hãy đổi sang L4 hoặc A100 "
              "(Runtime > Change runtime type).")
else:
    print("Không có GPU. Chỉ các bước chạy trên CPU / gọi API hoạt động được.")

### Khoá API (Colab Secrets)

Mở biểu tượng 🔑 (Secrets) ở thanh bên trái, thêm:

- `ANTHROPIC_API_KEY` = API key Claude của bạn (**bắt buộc** cho notebook này)
- `HF_TOKEN` = token Hugging Face (không bắt buộc)

và bật *Notebook access*. Nếu chưa thêm, ô dưới sẽ hỏi bạn nhập tay.

In [ ]:
import os
from getpass import getpass


def get_secret(name, required=False):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name]
    if required:
        return getpass(f"Nhập {name}: ")
    return None


os.environ["ANTHROPIC_API_KEY"] = get_secret("ANTHROPIC_API_KEY", required=True)
hf = get_secret("HF_TOKEN")  # không bắt buộc, giúp tải model nhanh hơn
if hf:
    os.environ["HF_TOKEN"] = hf
print("Đã nạp khoá:", [k for k in ("ANTHROPIC_API_KEY", "HF_TOKEN") if os.environ.get(k)])

## 1. Unit test (CPU)

In [ ]:
run("pytest chapter04/tests/ -q")

## 2. Few-shot ticket classifier (Listing 4.1)

In [ ]:
run("python -m chapter04.few_shot_demo --shots 8 --backend mock --output chapter04/runs/few_shot_mock.json")

Chạy với Qwen3-4B thật (GPU):

In [ ]:
run("python -m chapter04.few_shot_demo --shots 8 --backend hf --output chapter04/runs/few_shot_hf.json")

## 3. Many-shot prompt assembly (Listing 4.2)

In [ ]:
run('python -m chapter04.many_shot_demo --query "I cannot log in after the password reset email" --shots 20')
run('python -m chapter04.many_shot_demo --query "Refund my duplicate charge" --shots 10 --backend hash')

## 4. Prompt validator (Listing 4.3)

In [ ]:
run("python -m chapter04.prompt_validator --shots 8 --runs 3 --backend mock")
run("python -m chapter04.prompt_validator --shots 8 --runs 5 --backend hf --output chapter04/runs/validator_hf.json")

## 5. RAG tối giản (Listing 4.4)

In [ ]:
run('python -m chapter04.rag_minimal retrieve --query "How do I rotate my API key?" --k 3')
run('python -m chapter04.rag_minimal answer --query "What is the API key rotation grace period?" --k 3')

## 6. Đánh giá RAG: Precision@k / Hit@k / Hit@1 (Listing 4.5)

In [ ]:
run("python -m chapter04.scripts.listing_4_5_rag_eval --k 3")

In [ ]:
import json, pathlib
p = pathlib.Path("chapter04/runs/rag_eval.json")
print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False)[:2500] if p.exists() else f"Chưa có {p}")

## 7. (Bonus) RAG với Claude làm generator

Dùng lại `MinimalRAG` của sách, chỉ thay `stub_llm` bằng hàm gọi Claude.

In [ ]:
import anthropic
from chapter04.rag_minimal import MinimalRAG, make_st_embedder
from common.jsonl import read_jsonl

rag = MinimalRAG(embed=make_st_embedder())
rag.ingest(list(read_jsonl("chapter04/data/policy_docs.jsonl")))
client = anthropic.Anthropic()


def claude_llm(prompt: str) -> str:
    resp = client.messages.create(
        model="claude-opus-5-5",
        max_tokens=2000,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(b.text for b in resp.content if b.type == "text")


for q in ["What is the API key rotation grace period?",
          "How long do refunds take?",
          "What is the CEO's favourite colour?"]:   # câu không có trong tài liệu
    result = rag.answer(q, claude_llm, k=3)
    print("Q:", q)
    print("   nguồn:", [r["metadata"]["title"] for r in result["retrieved"]])
    print("   A:", result["answer"], "\n")